# Lecture 4 — Second-order methods and the statistics of the loss

**Week 2 · Day 4 · 45 min**

> **Headline.** Use the curvature. Newton converges in a handful of steps; its Hessian is
> the Fisher information; IRLS is how statisticians have been fitting GLMs for decades.
> Solve with Cholesky, never with an inverse.

Days 2 and 3 used the slope. Today we use the *curvature*, and three things fall out at
once: an algorithm whose cost does not depend on $\kappa$, a statistical interpretation
of its Hessian, and — for free — ridge regression.

**By the end of this lecture you can:**

1. derive Newton's step from a Taylor expansion and state why it converges quadratically;
2. write the GLM Hessian and recognize IRLS inside it;
3. explain why the Hessian is the Fisher information and what that buys you;
4. factor a matrix by Cholesky and say what a failure *means*;
5. explain why ridge makes the Hessian unconditionally invertible.

**You implement this afternoon:** `CholeskySolver`, `GLMLoss.hessian`, `NewtonDirection`,
`L2`, `RegularizedObjective`.

### Pacing

Target **41 min** of core material, hard cap **45 min**. Sections marked
*(cut first)* are the ones to drop if you are running behind; everything else is
load-bearing for the labwork. **The times below already include showing and discussing
the figures** — each figure is produced by the code cell above it, so run the notebook
once before the session.


> Figure 2 (the IRLS weights) is what makes §2 click, and Figure 4 is what the afternoon's
> damping lab is about — keep both. The cheapest cut inside a figure is Figure 3's middle
> panel; the eigenvalue lift on its left is the load-bearing half.


| § | Section | min |
|---|---|---|
| 1 | Newton's method  — *Figure 1* | 9 |
| 2 | The GLM Hessian, and a familiar face  — *Figure 2* | 9 |
| 2b | The Hessian is the Fisher information  *(cut first)* | 3 |
| 3 | Cholesky: solve, never invert | 7 |
| 4 | Ridge: the regularizer arrives, and it is free  — *Figure 3* | 7 |
| 5 | When Newton misbehaves  — *Figure 4* | 7 |
| 6 | Today's labs | 2 |
| | **total** | **44** |
| | **core only** | **41** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

rng = np.random.default_rng(4)

---

## 1. Newton's method

Gradient descent models $f$ by a *plane* through the current point. That model has no
notion of how far to go — hence the step size problem, and hence $\kappa$.

Use a better model. Taylor to second order around $x_k$:

$$f(x_k + p) \;\approx\; f(x_k) + g^\top p + \tfrac{1}{2}p^\top H p, \qquad g = \nabla f(x_k),\; H = \nabla^2 f(x_k)$$

This is a quadratic in $p$. Minimize it exactly: differentiate and set to zero,

$$g + Hp = 0 \qquad\Longrightarrow\qquad \boxed{H\,p = -g}$$

$p$ is the **Newton direction**. Two immediate consequences:

- If $f$ *is* a quadratic, the model is exact, and **one step lands on the minimum** —
  whatever $\kappa$ is. Compare the 691 steps from day 2.
- The step size is built into $p$. There is no $\alpha$ to tune, because the curvature
  already says how far to go. (We will add a line search anyway, for safety — see §5.)

### Quadratic convergence

Near a minimum with $H \succ 0$, Newton satisfies

$$\|x_{k+1} - x^\star\| \;\le\; C\,\|x_k - x^\star\|^2$$

The error is **squared** each step — the number of correct digits *doubles*. From
$10^{-2}$: $10^{-4}$, $10^{-8}$, $10^{-16}$. Three steps from two digits to machine
precision.

Contrast with day 2's *linear* convergence, where the error is merely multiplied by a
constant. This is a categorically different regime — and, like the heavy-ball rate
yesterday, it is a statement about the **tail**. Far from the optimum Newton can behave
badly, which is what §5 is about.

In [ ]:
# Newton on f(x) = e^x - 2x. Stationary point: f'= e^x - 2 = 0, so x* = log 2.
# Newton on f' = 0:  x <- x - f'/f'' = x - (e^x - 2)/e^x
x = 1.0
print(f"x* = log 2 = {np.log(2):.15f}\n")
print(f"{'k':>2} {'x_k':>18} {'error':>12} {'digits':>8}")
for k in range(6):
    err = abs(x - np.log(2))
    digits = -np.log10(err) if err > 0 else np.inf
    print(f"{k:2d} {x:18.15f} {err:12.3e} {digits:8.1f}")
    x = x - (np.exp(x) - 2) / np.exp(x)

Read the `digits` column: roughly $0.6 \to 1.4 \to 3.0 \to 6.2 \to 12.5$. **Each step
doubles it.** That is quadratic convergence, and it is what you are buying with the
second derivative.

---

## 2. The GLM Hessian, and a familiar face

Recall day 1: $L(w) = \frac{1}{n}\sum_i \varphi(z_i, y_i)$ with $z = Xw$, and
$\nabla L = \frac{1}{n}X^\top \varphi'(z, y)$.

Differentiate once more. By the chain rule, $\partial z_i/\partial w = x_i$, so

$$\nabla^2 L(w) \;=\; \frac{1}{n}\sum_i \varphi''(z_i, y_i)\, x_i x_i^\top
\;=\; \boxed{\frac{1}{n} X^\top D X}, \qquad D = \mathrm{diag}\big(\varphi''(z_i, y_i)\big)$$

**One formula, every GLM** — the same structural economy as the gradient. Only $D$
changes:

| Model | $\varphi''(z,y)$ | $D$ | Consequence |
|---|---|---|---|
| linear | $1$ | $I$ | $H = X^\top X/n$, constant. One Newton step **is** the normal equations |
| logistic | $\sigma(z)(1-\sigma(z))$ | $\mathrm{diag}(\sigma(1-\sigma))$ | weights depend on $w$, so re-weight each iteration |
| Poisson | $e^{z}$ | $\mathrm{diag}(e^z)$ | same structure again |

### This is IRLS

For logistic regression the Newton step solves

$$\frac{1}{n}X^\top D X\, p \;=\; -\frac{1}{n}X^\top(\sigma(z) - y)$$

which is precisely a **weighted least-squares problem** — the normal equations for
regressing a working response on $X$ with weights $D$. Since $D$ depends on $w$, you
re-weight and re-solve each iteration. This algorithm is called **iteratively reweighted
least squares**, and it is how `glm()` in R and every statistics package fits GLMs.

You are not implementing a numerical-analysis curiosity. You are implementing the
standard tool of applied statistics, and you have derived it from Newton's method.

In [ ]:
# Newton in one dimension: replace f by a parabola, jump to the parabola's minimum.
fN  = lambda x: np.exp(x) - 2 * x
fN1 = lambda x: np.exp(x) - 2
fN2 = lambda x: np.exp(x)
x_star = np.log(2)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

xs = np.linspace(0.0, 1.5, 400)
ax[0].plot(xs, fN(xs), lw=2.4, color="k", label=r"$f(x) = e^x - 2x$", zorder=5)

xk = 1.0
for k, col in [(0, "crimson"), (1, "tab:blue")]:
    g_, h_ = fN1(xk), fN2(xk)
    model = fN(xk) + g_ * (xs - xk) + 0.5 * h_ * (xs - xk) ** 2
    x_next = xk - g_ / h_
    ax[0].plot(xs, model, lw=1.6, ls="--", color=col,
               label=f"quadratic model at $x_{k}$")
    ax[0].plot(xk, fN(xk), "o", color=col, ms=9, zorder=6)
    ax[0].annotate(f"$x_{k}$", xy=(xk, fN(xk)), xytext=(0, 11),
                   textcoords="offset points", fontsize=9, color=col, ha="center")
    ax[0].plot(x_next, fN(xk) + g_ * (x_next - xk) + 0.5 * h_ * (x_next - xk) ** 2,
               "v", color=col, ms=9, zorder=6)
    ax[0].annotate("", xy=(x_next, fN(xk) + g_ * (x_next - xk) + 0.5 * h_ * (x_next - xk) ** 2),
                   xytext=(xk, fN(xk)),
                   arrowprops=dict(arrowstyle="->", color=col, lw=1.6))
    xk = x_next

ax[0].axvline(x_star, color="0.4", ls=":", lw=1.2)
ax[0].annotate("$x^\\star = \\log 2$", xy=(x_star, 1.54), fontsize=8, color="0.4",
               ha="center")
ax[0].set_xlabel("$x$"); ax[0].set_ylabel("value")
ax[0].set_title("Each step minimizes a parabola, not $f$", fontsize=9)
ax[0].legend(fontsize=8, loc="upper center"); ax[0].set_ylim(0.58, 1.62)

# --- right: doubling digits versus a constant gain per step.
def digits_of(errs):
    return -np.log10(np.maximum(np.asarray(errs), 1e-17))

xn, e_newton = 1.0, []
for _ in range(7):
    e_newton.append(abs(xn - x_star))
    xn = xn - fN1(xn) / fN2(xn)

# Gradient descent must pick alpha from an upper bound on the curvature it may MEET
# (alpha = 1/L, day 2), not from the curvature at the optimum -- which it does not know.
xg, e_gd = 1.0, []
L_gd = fN2(1.0)                           # f'' is largest at the starting point here
alpha_gd = 1.0 / L_gd
for _ in range(40):
    e_gd.append(abs(xg - x_star))
    xg = xg - alpha_gd * fN1(xg)

ax[1].plot(digits_of(e_newton), "o-", color="crimson", lw=2.0, ms=7,
           label="Newton (uses $f''$ at $x_k$)")
ax[1].plot(digits_of(e_gd), "s-", color="tab:blue", lw=1.8, ms=4,
           label=f"gradient descent ($\\alpha = 1/L$ = {alpha_gd:.2f})")
ax[1].axhline(16, color="0.4", ls=":", lw=1.2)
ax[1].annotate("machine precision", xy=(39, 16.4), fontsize=8, color="0.4", ha="right")
ax[1].set_xlabel("iteration $k$"); ax[1].set_ylabel("correct digits  $-\\log_{10}|x_k-x^\\star|$")
ax[1].set_title("Newton doubles the digits; descent adds a constant", fontsize=9)
ax[1].set_xlim(-0.5, 40); ax[1].set_ylim(0, 18)
ax[1].legend(fontsize=8, loc="lower right")
gain = np.diff(digits_of(e_gd))[5]
ax[1].annotate(f"a steady {gain:.2f} digits\nper step", xy=(20, digits_of(e_gd)[20]),
               xytext=(-10, 30), textcoords="offset points", fontsize=8, color="tab:blue",
               ha="center", arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.1))

plt.tight_layout()
plt.show()

print("Newton, correct digits per step:", np.round(digits_of(e_newton), 2))

**Figure 1 — what "use the curvature" buys.**

*Left:* the black curve is $f$. At $x_0 = 1$ we build the dashed red parabola — it matches
$f$ in value, slope **and** curvature at that point — and jump straight to its minimum.
That lands us at $x_1$, where we build a new parabola (blue) and jump again. By $x_2$ the
parabola and $f$ are indistinguishable near the minimum, which is exactly why the method
then converges so violently fast.

Compare gradient descent, which models $f$ by a *straight line*. A line has no minimum, so
a first-order method has nothing to tell it how far to go — and that missing information
is the step-size problem of day 2, and the factor $\kappa$.

*Right:* the same two methods, counting correct digits. Gradient descent adds a roughly
**constant** number of digits per step — the straight line is day 2's linear convergence.
Newton's curve bends upward, because the digit count **doubles**:
$0.6 \to 1.4 \to 3.0 \to 6.2 \to 12.5$, then machine precision. Four useful steps in
total, against roughly forty.

The comparison is set up carefully, and the setup is the lesson. Gradient descent is given
$\alpha = 1/L$ with $L$ an upper bound on the curvature it may meet — which is the best it
can do, because choosing a step requires knowing a curvature and a first-order method has
none. Hand it instead the single number $1/f''(x^\star)$ and it would land on the optimum
in **one** step. But $x^\star$ is what we are looking for, and the method that estimates
that curvature as it goes, at the current point rather than at the unknown optimum, is
exactly Newton's.

> **Both statements are about the tail.** Newton earns this behaviour only once it is
> close enough that the quadratic model is trustworthy. §5 shows what it does before then,
> and the answer is not reassuring.

### The Hessian is the Fisher information

One more step, and optimization turns into statistics.

For a model at its maximum-likelihood estimate, the matrix

$$\mathcal{I}(w) \;=\; n\,\nabla^2 L(w) \;=\; X^\top D X$$

is the **observed Fisher information** — literally, how much information the data carries
about $w$. Standard asymptotic theory then gives

$$\widehat{\mathrm{cov}}(\hat w) \;\approx\; \mathcal{I}(\hat w)^{-1} = (X^\top D X)^{-1}$$

So the very matrix you factor to *take a step* also gives you the **standard errors** of
your estimates. The diagonal of its inverse, square-rooted, is what a statistics package
prints next to each coefficient.

This is worth pausing on, because it makes two apparently unrelated facts the same fact:

| Optimization statement | Statistical statement |
|---|---|
| $H$ is nearly singular | the parameters are barely identified by the data |
| $H$ has a zero eigenvalue | a direction in $w$ the data cannot distinguish at all |
| "Newton is ill-conditioned here" | "these standard errors are enormous" |
| Cholesky **fails** | the information matrix is not positive definite — no unique MLE |

A hard optimization problem and an uninformative dataset are, very often, the same
problem seen from two sides.

In [ ]:
# Hessian of a GLM, and the standard errors that fall out of it.
n, p = 400, 3
X = np.column_stack([np.ones(n), rng.normal(size=(n, p - 1))])
w_true = np.array([0.5, -1.0, 2.0])
sigmoid = lambda z: 1 / (1 + np.exp(-z))
y = (rng.random(n) < sigmoid(X @ w_true)).astype(float)

def gradient(w):
    return X.T @ (sigmoid(X @ w) - y) / n

def hessian(w):
    s = sigmoid(X @ w)
    return X.T @ (X * (s * (1 - s))[:, None]) / n       # X.T D X / n

# Newton, using the Hessian. No step size anywhere.
w = np.zeros(p)
print(f"{'iter':>5} {'|grad|':>12}   w")
for k in range(7):
    g = gradient(w)
    print(f"{k:5d} {np.linalg.norm(g):12.3e}   {w}")
    w = w - np.linalg.solve(hessian(w), g)

print(f"\ntrue w        : {w_true}")
se = np.sqrt(np.diag(np.linalg.inv(n * hessian(w))))
print(f"standard errors: {se}      <- from the SAME matrix we just solved with")

In [ ]:
# The weights D that IRLS re-computes every iteration -- and where the information is.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

zz = np.linspace(-8, 8, 400)
ww = sigmoid(zz) * (1 - sigmoid(zz))
ax[0].plot(zz, ww, lw=2.4, color="tab:purple")
ax[0].fill_between(zz, 0, ww, color="tab:purple", alpha=0.15)
ax[0].plot(0, 0.25, "o", color="tab:purple", ms=9, zorder=6)
ax[0].annotate("max = 0.25 at $z$ = 0\n(model says 50/50)", xy=(0, 0.25), xytext=(0, -46),
               textcoords="offset points", fontsize=8, ha="center",
               arrowprops=dict(arrowstyle="->", lw=1.1))
for zc in (-5, 5):
    ax[0].annotate(f"$\\sigma(1-\\sigma)$ = {sigmoid(zc)*(1-sigmoid(zc)):.4f}\n"
                   "confident → ignored",
                   xy=(zc, sigmoid(zc) * (1 - sigmoid(zc))),
                   xytext=(0, 60), textcoords="offset points", fontsize=7.5, ha="center",
                   arrowprops=dict(arrowstyle="->", lw=1.0))
ax[0].set_xlabel("$z = x^\\top w$  (the model's logit)")
ax[0].set_ylabel("weight $\\varphi''(z) = \\sigma(z)(1-\\sigma(z))$")
ax[0].set_title("The logistic weight: information lives at the boundary", fontsize=9)
ax[0].set_ylim(0, 0.34)

# --- right: the same weights, on the actual data.
w_fit = np.zeros(p)
for _ in range(8):
    w_fit = w_fit - np.linalg.solve(hessian(w_fit), gradient(w_fit))
s_fit = sigmoid(X @ w_fit)
wt = s_fit * (1 - s_fit)

sc = ax[1].scatter(X[:, 1], X[:, 2], c=wt, cmap="viridis", s=26,
                   vmin=0, vmax=0.25, edgecolors="none")
xb = np.linspace(X[:, 1].min(), X[:, 1].max(), 2)
ax[1].plot(xb, -(w_fit[0] + w_fit[1] * xb) / w_fit[2], "r--", lw=2.0,
           label="fitted boundary  $x^\\top w = 0$")
plt.colorbar(sc, ax=ax[1], label="weight in the Hessian")
ax[1].set_xlabel("$x_1$"); ax[1].set_ylabel("$x_2$")
ax[1].set_title("Bright points build the Hessian. Dark ones barely count.", fontsize=9)
ax[1].legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()

share = np.sort(wt)[::-1]
print(f"the top 25% of points carry {share[:n//4].sum()/share.sum():.0%} of the total weight")
print(f"the bottom 25% carry        {share[-(n//4):].sum()/share.sum():.0%}")

**Figure 2 — why it is called *reweighted* least squares.**

*Left:* the weight $\varphi''(z) = \sigma(z)(1-\sigma(z))$ that sits on the diagonal of
$D$. It peaks at $0.25$ where the model is maximally uncertain ($z = 0$, predicted
probability $\tfrac12$) and collapses toward zero as $|z|$ grows. A sample the model is
already confident about contributes **almost nothing** to $X^\top D X$.

*Right:* those weights painted onto the data. The bright band hugs the fitted decision
boundary; points far from it are nearly black. And since $D$ depends on $w$, the
brightness map is redrawn at every iteration — that is the "iteratively reweighted" part,
and it is why the logistic Hessian must be rebuilt each step while the linear one
($D = I$) never changes.

> **This is the Fisher information made visual.** The matrix that tells you how far to
> step is the same matrix that tells you how much the data knows, and both say: the
> information is concentrated near the boundary. A dataset whose points all sit far from
> the boundary is *easy to classify* and *hard to fit precisely* — a large $\|w\|$ and a
> tiny Hessian are the same observation, which is exactly the separable-data failure in
> §4.

Six or seven iterations, from $\|g\| \sim 10^{-1}$ to machine zero — and the standard
errors come from the matrix we were factoring anyway.

---

## 3. Cholesky: solve, never invert

The Newton step needs $p$ with $Hp = -g$. **Do not compute $H^{-1}$.**

| | flops | numerics |
|---|---|---|
| form $H^{-1}$, then multiply | $\sim n^3$ | worse — inversion amplifies error |
| factor $H = LL^\top$, two triangular solves | $\sim n^3/3$ | better, and stable |

Three times cheaper and more accurate. There is never a reason to invert. (The only
exception on this course: the covariance $\mathcal{I}^{-1}$ above, where the inverse
itself is the quantity you want — and even then you get it from the factorization.)

### The factorization

For symmetric positive definite $A$ there is a unique lower-triangular $L$ with positive
diagonal such that $A = LL^\top$. Comparing entries:

$$L_{jj} = \sqrt{A_{jj} - \sum_{k<j} L_{jk}^2}, \qquad
L_{ij} = \frac{1}{L_{jj}}\Big(A_{ij} - \sum_{k<j} L_{ik}L_{jk}\Big) \quad (i > j)$$

Then solve $Ax = b$ in two sweeps: $Ly = b$ forward, $L^\top x = y$ backward.

### The failure is the feature

Look at the diagonal formula: it takes a **square root**. If the quantity under it is
$\le 0$, the factorization cannot proceed — and that happens **exactly** when $A$ is not
positive definite.

So Cholesky is simultaneously a solver and a definiteness test, at no extra cost. Do not
paper over the failure with a fallback: raise `NotPositiveDefiniteError` and let the
caller decide. A failure is information — it says the quadratic model has a direction of
non-positive curvature, i.e. we are at or near a saddle.

In [ ]:
def cholesky(A):
    """A = L L^T. Raises if A is not positive definite."""
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.zeros((n, n))
    for j in range(n):
        d = A[j, j] - L[j, :j] @ L[j, :j]
        if d <= 0:
            raise ValueError(f"not positive definite: pivot {j} would be sqrt({d:.4f})")
        L[j, j] = np.sqrt(d)
        if j + 1 < n:                                   # vectorized over i > j
            L[j + 1:, j] = (A[j + 1:, j] - L[j + 1:, :j] @ L[j, :j]) / L[j, j]
    return L


A = np.array([[4.0, 2.0, 2.0], [2.0, 5.0, 3.0], [2.0, 3.0, 6.0]])
L = cholesky(A)
print("A =\n", A)
print("\nL =\n", L, "   <- check this by hand: [[2,0,0],[1,2,0],[1,1,2]]")
print("\nL @ L.T - A  (should be 0):\n", L @ L.T - A)

In [ ]:
# The matrix from Lecture 1. Eigenvalues 3 and -1, so it must fail.
B = np.array([[1.0, 2.0], [2.0, 1.0]])
print("eigenvalues:", np.linalg.eigvalsh(B))
try:
    cholesky(B)
except ValueError as e:
    print("cholesky ->", e)
    print("\nThe second pivot would need sqrt(1 - 4) = sqrt(-3). The failure is not a")
    print("numerical accident; it is the matrix telling you it is indefinite.")

---

## 4. Ridge: the regularizer arrives, and it is free

So far the penalty has been a promise from day 1. Today it becomes an object.

$$r(w) = \tfrac{\lambda}{2}\|w\|^2, \qquad \nabla r = \lambda w, \qquad \nabla^2 r = \lambda I$$

Define `RegularizedObjective(objective, regularizer)`, an **adapter** presenting
$f + r$ as a single `IObjective`. Then

$$\text{ridge regression} \;=\; \texttt{RegularizedObjective(GLMLoss, L2)} \text{ handed to any optimizer you already wrote.}$$

**Not a new optimizer. Not a new loss. Not a new anything.** Gradient descent, momentum,
SGD, Adam and Newton all gain a regularized version the moment this one adapter exists.
That is the open/closed principle with a number attached: one new class, nine new
capabilities.

### And it fixes the conditioning

$$\nabla^2(f + r) = H + \lambda I$$

Adding $\lambda I$ shifts **every eigenvalue up by $\lambda$**. So for $\lambda > 0$ the
result is positive definite no matter what $H$ was — Cholesky can no longer fail. And the
condition number improves:

$$\kappa(H + \lambda I) = \frac{L + \lambda}{\mu + \lambda} \;<\; \frac{L}{\mu} = \kappa(H)$$

Remember this trick. Tomorrow Levenberg–Marquardt is built from exactly it.

### What it means statistically

Day 1: ridge is a Gaussian prior. Here is the payoff. On **separable** data (a hyperplane
perfectly splits the classes) the logistic MLE does not exist — pushing $\|w\| \to \infty$
drives the loss to zero, so there is no finite minimizer and Newton marches off to
infinity. The prior supplies the missing information and pins the solution down.

In [ ]:
# Separable data: the unregularized logistic MLE DOES NOT EXIST.
Xs = np.array([[1.0, -2.0], [1.0, -1.0], [1.0, 1.0], [1.0, 2.0]])
ys = np.array([0.0, 0.0, 1.0, 1.0])       # perfectly split at 0


def fit(lam, n_iter):
    """Newton on the ridge-penalized logistic loss. Returns (w, iterations completed)."""
    w = np.zeros(2)
    for k in range(n_iter):
        s = sigmoid(Xs @ w)
        g = Xs.T @ (s - ys) / len(ys) + lam * w
        H = Xs.T @ (Xs * (s * (1 - s))[:, None]) / len(ys) + lam * np.eye(2)
        try:
            w = w - np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            return w, k                    # the Hessian collapsed to singular
    return w, n_iter


print("lambda = 0 : the norm just grows, and the Hessian degenerates with it")
for n_iter in [5, 10, 20, 40]:
    w, done = fit(0.0, n_iter)
    s = sigmoid(Xs @ w)
    H = Xs.T @ (Xs * (s * (1 - s))[:, None]) / len(ys)
    note = "" if done == n_iter else f"   <- Cholesky/solve FAILED at iteration {done}"
    print(f"  after {n_iter:3d} Newton steps: |w| = {np.linalg.norm(w):9.3f}, "
          f"smallest eigenvalue of H = {np.linalg.eigvalsh(H).min():.2e}{note}")

print("\nlambda > 0 : the prior supplies the missing information, and H is safe")
for lam in [1e-3, 1e-2, 1e-1]:
    w, _ = fit(lam, 60)
    s = sigmoid(Xs @ w)
    H = Xs.T @ (Xs * (s * (1 - s))[:, None]) / len(ys) + lam * np.eye(2)
    print(f"  lambda = {lam:6.3f}  ->  |w| = {np.linalg.norm(w):8.4f}, "
          f"smallest eigenvalue of H = {np.linalg.eigvalsh(H).min():.2e}")

print("\nEvery eigenvalue is lifted by exactly lambda. That is the whole trick,")
print("and tomorrow Levenberg-Marquardt uses it again.")

In [ ]:
# What lambda does: to the eigenvalues, to kappa, and to the solution on separable data.
# Use a CORRELATED design -- two nearly duplicated features -- so there is a genuinely
# small eigenvalue to look at. (The tidy data above has kappa ~ 3; nothing to see.)
u = rng.normal(size=n)
X_ill = np.column_stack([np.ones(n), u, u + 0.03 * rng.normal(size=n)])
y_ill = (rng.random(n) < sigmoid(X_ill @ w_true)).astype(float)

w_ml = np.zeros(p)
for _ in range(12):
    s_ = sigmoid(X_ill @ w_ml)
    H_ = X_ill.T @ (X_ill * (s_ * (1 - s_))[:, None]) / n
    w_ml = w_ml - np.linalg.solve(H_, X_ill.T @ (s_ - y_ill) / n)
s_ = sigmoid(X_ill @ w_ml)
H0 = X_ill.T @ (X_ill * (s_ * (1 - s_))[:, None]) / n
ev0 = np.linalg.eigvalsh(H0)
lams = np.logspace(-6, 1, 200)
print(f"correlated design: eigenvalues of H = {np.round(ev0, 6)}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))

for i, e in enumerate(ev0):
    ax[0].loglog(lams, e + lams, lw=1.9,
                 label=f"$\\mu_{i+1} + \\lambda$   ($\\mu_{i+1}$ = {e:.2e})")
ax[0].loglog(lams, lams, "k:", lw=1.3, label=r"$\lambda$ (the asymptote)")
ax[0].set_xlabel(r"$\lambda$"); ax[0].set_ylabel("eigenvalue of $H + \\lambda I$")
ax[0].set_title("Every eigenvalue is lifted by exactly $\\lambda$", fontsize=9)
ax[0].legend(fontsize=7.5, loc="upper left")

ax[1].loglog(lams, (ev0.max() + lams) / (ev0.min() + lams), lw=2.2, color="crimson")
ax[1].axhline(ev0.max() / ev0.min(), color="0.35", ls="--", lw=1.2)
ax[1].annotate(f"$\\kappa(H)$ = {ev0.max()/ev0.min():,.0f}", xy=(1.5e-6, ev0.max()/ev0.min()),
               xytext=(0, 7), textcoords="offset points", fontsize=8, color="0.35")
ax[1].axhline(1.0, color="0.35", ls=":", lw=1.2)
ax[1].annotate(r"$\kappa \to 1$: a perfect bowl," "\n" r"but of the WRONG function",
               xy=(2e-5, 2.2), fontsize=8, color="0.35", ha="left", va="bottom",
               bbox=dict(fc="white", ec="0.6", lw=0.8, alpha=0.9))
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel(r"$\kappa(H + \lambda I)$")
ax[1].set_title("Conditioning improves monotonically with $\\lambda$", fontsize=9)

# --- the separable problem: without a prior there is no finite answer.
lam_s = np.logspace(-8, 0, 41)
norms = np.array([np.linalg.norm(fit(lm, 400)[0]) for lm in lam_s])
ax[2].semilogx(lam_s, norms, "o-", color="tab:blue", lw=1.9, ms=4)
ax[2].set_xlabel(r"$\lambda$"); ax[2].set_ylabel(r"$\|w_\lambda\|$")
ax[2].set_title("Separable data: $\\|w\\|$ grows without bound as $\\lambda \\to 0$",
                fontsize=9)
ax[2].annotate("a straight line on a log axis:\n"
               f"every decade of $\\lambda$ adds\n~{(norms[0]-norms[-1])/8:.1f} to $\\|w\\|$, forever.\n"
               "There is no finite MLE.",
               xy=(3e-7, norms[4]), xytext=(10, -62), textcoords="offset points",
               fontsize=8, color="tab:blue",
               bbox=dict(fc="white", ec="tab:blue", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.1))
ax[2].set_ylim(0, norms.max() * 1.15)

plt.tight_layout()
plt.show()

print(f"kappa(H)           = {ev0.max()/ev0.min():10.1f}")
print(f"kappa(H + 1e-3 I)  = {(ev0.max()+1e-3)/(ev0.min()+1e-3):10.1f}")
print(f"kappa(H + 1e-2 I)  = {(ev0.max()+1e-2)/(ev0.min()+1e-2):10.1f}")
print(f"kappa(H + 1.0  I)  = {(ev0.max()+1.0)/(ev0.min()+1.0):10.1f}")

**Figure 3 — one parameter, three simultaneous effects.**

The first two panels use a **correlated design**: two of the three features are almost
duplicates of each other, which is what real data looks like when two columns measure
nearly the same thing. That gives $H$ one genuinely tiny eigenvalue — the direction in
$w$ that trades one feature against its twin, which the data can barely distinguish.

*Left:* the spectrum of $H + \lambda I$. Each eigenvalue is just $\mu_i + \lambda$, so
every curve is flat while $\lambda \ll \mu_i$ and then joins the dotted diagonal. The
**smallest** eigenvalue is the one that matters: it is what Cholesky's last pivot tests,
and $\lambda$ puts a floor under it. For $\lambda > 0$ the matrix is positive definite
whatever $H$ was — even if $H$ had a negative eigenvalue, which is how the same trick will
rescue Gauss–Newton tomorrow.

*Middle:* the consequence for $\kappa$. It falls monotonically from $\kappa(H)$ toward
$1$. Note the warning in the label: as $\lambda$ grows you are solving an ever easier
problem that is ever further from the one you were asked about. $\lambda$ is not free
conditioning — it is a trade, and the currency is bias.

*Right:* the case where the trade is not optional. On perfectly separable data the
likelihood has no finite maximizer — pushing $\|w\|\to\infty$ makes every prediction
certain and the loss zero. Plotted against $\log\lambda$ the norm is a **straight rising
line**: it never levels off, so there is no limiting solution to converge to. Any
$\lambda > 0$ picks out a finite, unique answer; $\lambda = 0$ picks out nothing at all.

> **Three readings of the same $\lambda$.** *Numerical:* it makes Cholesky succeed.
> *Optimization:* it lowers $\kappa$, so every method converges faster. *Statistical:* it
> is a Gaussian prior supplying the information the data lacks. They are not three
> benefits that happen to coincide — they are one fact stated three ways.

---

## 5. When Newton misbehaves

Newton's quadratic convergence is **local**. Away from a minimum, two quite different
things go wrong, and they need two quite different repairs. Keeping them apart is the
point of this section.

**Failure A — the step is too long.** $H$ is positive definite, so the direction is fine,
but the full step $\alpha = 1$ overshoots because the quadratic model is only accurate
nearby. This is a *length* problem.

*Repair:* use the Newton **direction** with an Armijo **step**, starting at $\alpha = 1$.
Near the optimum $\alpha = 1$ is accepted immediately and you keep quadratic convergence;
far away, backtracking pulls you back. Note the architecture: this is `NewtonDirection` +
`Armijo` in the day-2 loop. **Three existing objects, zero new code.**

**Failure B — the direction itself is wrong.** $H$ is indefinite. Then $p$ solving
$Hp = -g$ is not a descent direction at all, and Newton will converge happily to a saddle
or a maximum, because $\nabla f = 0$ is the only condition it checks. This is a
*direction* problem, and **the line search cannot fix it.**

That last claim is worth checking rather than believing. On the double well below, at
$x_0 = 0.3$: $g = -1.092$, $H = -2.92$, so $p = -g/H = -0.374$ and the directional
derivative is
$$g^\top p = +0.408 > 0,$$
an **ascent** direction. Armijo asks for a *decrease* along $p$; there is none to find, so
it backtracks its whole budget and gives up, leaving the iterate exactly where it started.
And with your own `CholeskySolver` it never even gets that far — Cholesky refuses the
indefinite matrix and raises `NotPositiveDefiniteError` before any line search runs.

So a line search is a genuine safeguard — it stops you taking the bad step — but it is
only ever an *auditor*. It can veto a direction; it cannot supply a better one.

*Repair:* change the direction. If Cholesky fails, solve $(H + \tau I)p = -g$ instead,
doubling $\tau$ until it succeeds. As $\tau$ grows the step rotates from the Newton
direction toward steepest descent and shortens, so large $\tau$ degrades gracefully
rather than failing. You have seen $H + \tau I$ twice now — as ridge, and as this.
Tomorrow it appears a third time, with a principled rule for choosing $\tau$.


In [ ]:
# A double well: f(x) = (x^2 - 1)^2. Minima at +-1, a local MAXIMUM at 0.
f  = lambda x: (x ** 2 - 1) ** 2
f1 = lambda x: 4 * x * (x ** 2 - 1)
f2 = lambda x: 12 * x ** 2 - 4

x0 = 0.3
print(f"f''({x0}) = {f2(x0):.3f} < 0  -- negative curvature, so the model is a downward parabola\n")

x = x0
for k in range(4):
    print(f"  pure Newton   k={k}  x={x: .6f}  f={f(x):.6f}")
    x = x - f1(x) / f2(x)
print("  -> converging to x = 0, which is a MAXIMUM. Newton only knows f' = 0.\n")

x, tau = x0, 0.0
for k in range(6):
    h = f2(x)
    tau = 0.0 if h > 0 else abs(h) + 1.0            # force positive curvature
    x = x - f1(x) / (h + tau)
print(f"  damped (H + tau I): x = {x:.6f}, f = {f(x):.6e}   <- a real minimum")

In [ ]:
# Why an undamped Newton step is not safe far from a minimum.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))
xs = np.linspace(-1.5, 1.7, 500)

for a_ in ax:
    a_.plot(xs, f(xs), lw=2.4, color="k", zorder=5)
    a_.axhline(0, color="0.8", lw=0.8)
    for m in (-1, 1):
        a_.plot(m, 0, "*", color="gold", ms=17, mec="k", mew=0.8, zorder=7)

# --- left: the model at x0 = 0.3 is an UPSIDE-DOWN parabola.
x0 = 0.3
g0, h0 = f1(x0), f2(x0)
model = f(x0) + g0 * (xs - x0) + 0.5 * h0 * (xs - x0) ** 2
ax[0].plot(xs, model, "--", lw=1.9, color="crimson", label=f"model at $x_0$ = {x0}")
ax[0].plot(x0, f(x0), "o", color="crimson", ms=9, zorder=8)
ax[0].annotate(f"$f''({x0})$ = {h0:.2f} < 0\n→ the model opens DOWNWARD,\n"
               "so its stationary point is a MAXIMUM",
               xy=(x0, f(x0)), xytext=(-8, -74), textcoords="offset points", fontsize=8,
               color="crimson", ha="center",
               bbox=dict(fc="white", ec="crimson", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.2))
ax[0].set_ylim(-1.4, 2.2); ax[0].set_xlabel("$x$"); ax[0].set_ylabel("$f(x)$")
ax[0].set_title("$f(x) = (x^2-1)^2$ and Newton's model at $x_0$", fontsize=9)
ax[0].legend(fontsize=8, loc="upper center")

# --- right: where each variant actually ends up.
xp, pure = x0, [x0]
for _ in range(6):
    xp = xp - f1(xp) / f2(xp)
    pure.append(xp)

xd, damped = x0, [x0]
for _ in range(6):
    h = f2(xd)
    tau = 0.0 if h > 0 else abs(h) + 1.0
    xd = xd - f1(xd) / (h + tau)
    damped.append(xd)

ax[1].plot(pure, [f(v) for v in pure], "o--", color="crimson", ms=7, lw=1.4,
           label=f"pure Newton → {pure[-1]:.3f}  (a MAXIMUM)", zorder=8)
ax[1].plot(damped, [f(v) for v in damped], "s--", color="tab:blue", ms=7, lw=1.4,
           label=f"damped $H+\\tau I$ → {damped[-1]:.3f}  (a minimum)", zorder=9)
ax[1].plot(0, f(0), "X", color="crimson", ms=15, mec="k", mew=0.8, zorder=10)
ax[1].annotate("$\\nabla f = 0$ here too!", xy=(0, 1.0), xytext=(0, 26),
               textcoords="offset points", fontsize=8, color="crimson", ha="center",
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.1))
ax[1].set_ylim(-0.35, 2.2); ax[1].set_xlabel("$x$"); ax[1].set_ylabel("$f(x)$")
ax[1].set_title("Same start, same function, two outcomes", fontsize=9)
ax[1].legend(fontsize=8, loc="upper center")

plt.tight_layout()
plt.show()

**Figure 4 — Newton finds stationary points, not minima.**

*Left:* at $x_0 = 0.3$ the curvature is **negative**, $f''(0.3) = -2.92$. The quadratic
model built there is therefore an upside-down parabola (dashed red), and "minimize the
model" is meaningless — solving $Hp = -g$ finds its *stationary* point, which here is its
maximum. Newton has no way to notice: the equation $g + Hp = 0$ does not care about the
sign of $H$.

*Right:* the consequence. Pure Newton (red) walks to $x = 0$ — the local **maximum** of
$f$, sitting between the two real minima — and is perfectly happy there, because
$\nabla f(0) = 0$ and that is the only condition it checks. The damped version (blue),
which replaces $H$ by $H + \tau I$ until the curvature is positive, sets off in the
genuinely downhill direction and lands on the minimum at $x = 1$.

> **Do not read this figure as an argument for the line search.** It is tempting to say
> "Armijo would have rejected the red step because it increases $f$", and Armijo *would*
> reject it — but rejecting is all it can do. As §5 showed, the Newton direction here is
> an ascent direction, so every backtrack is also uphill; Armijo exhausts its budget and
> the iterate never moves. You trade converging to the wrong answer for not converging at
> all, which is better, but it is not a solution.
>
> The blue curve exists because $H$ was *replaced*, not because the step was shortened.
> In Lab 3 you will run all three — undamped, `Armijo`-only, and `ModifiedNewton` — and
> see that only the third one reaches $x = 1$.


---

## 6. Today's labs

| Lab | What | The point |
|---|---|---|
| 1 (55 min) | `cholesky`, `solve_lower`, `solve_upper_from_lower`, `CholeskySolver` | one solver, reused three times this week |
| 2 (55 min) | `GLMLoss.hessian`, `check_hessian`, `NewtonDirection`, `newton()` | linear model solved in **1** iteration; **no edit to the loop** |
| 3 (30 min) | damping and failure | the double well; `H + \tau I` |
| 4 (20 min) | `L2`, `NoRegularizer`, `RegularizedObjective` | ridge for free — live open/closed |

Note `check_hessian` in Lab 2: a Hessian **is** the Jacobian of the gradient, so day 1's
`numerical_jacobian` already does the work. Do not write a second finite-difference
routine.

**Two questions for the debrief:**

1. Why not just invert $H$? Give both the flop count and the numerical reason.
2. What does "Cholesky failed" mean **geometrically**, and what does it mean
   **statistically**? They are the same sentence in two languages.

> **Tomorrow.** Newton needs the full Hessian, which is $O(np^2)$ to build. But when the
> loss is a sum of squared residuals there is a much cheaper approximation — accurate
> when the residuals are small, and dangerous when they are not. That is Gauss–Newton,
> and repairing its failure mode with today's $H + \tau I$ gives Levenberg–Marquardt.